In [0]:

# Create a Data Frame with a few deliberate duplicates
from pyspark.sql.functions import col, count
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

data = [(1, "Alice", "IT", 55000),
        (2, "Ben", "HR", 48000),
        (3, "Cara", "IT", 62000),
        (2, "Ben", "HR", 48000),      # exact duplicate of row 2
        (4, "Dev", "Finance", 45000),
        (1, "Alice", "IT", 55000)]    # exact duplicate of row 1
columns = ["id", "name", "department", "salary"]

employees = spark.createDataFrame(data, columns)
employees.display()

In [0]:
#Knowing THAT Duplicates Exist — the Day 6 Pattern
from pyspark.sql.functions import col
duplicate_ids = employees.groupBy("id").count().filter(col("count")> 1)
duplicate_ids.display()

In [0]:
#. Knowing WHICH Rows Are the Duplicates — the Day 8 Pattern
window_spec = Window.partitionBy("id").orderBy("id")
with_row_num = employees.withColumn("row_num", row_number().over(window_spec))
with_row_num.display()

duplicates_to_remove = with_row_num.filter(col("row_num") > 1)
duplicates_to_remove.display()

In [0]:
#Removing Duplicates, Keeping the First Occurrence
deduplicated = with_row_num.filter(col("row_num") == 1).drop("row_num")
deduplicated.display()
print("Original:", employees.count(), "| After dedup:", deduplicated.count())

In [0]:
#The Built-In Shortcut: dropDuplicates()
employees.dropDuplicates().display()
# compares ALL columns by default - only removes rows that are 100% identical across every field
employees.dropDuplicates(["id"]).display()
# only considers "id" - removes a row if its id has already been seen, regardless of other columns


In [0]:
# Create orders DataFrame with deliberate duplicates
from datetime import datetime

orders_data = [
    (101, "2024-01-15", "Alice", 250.00, "Shipped"),
    (102, "2024-01-16", "Bob", 150.00, "Delivered"),
    (103, "2024-01-17", "Carol", 320.00, "Processing"),
    (101, "2024-01-14", "Alice", 250.00, "Shipped"),     # duplicate order_id, older date
    (104, "2024-01-18", "Dave", 175.00, "Shipped"),
    (102, "2024-01-16", "Bob", 150.00, "Delivered"),     # exact duplicate
    (105, "2024-01-19", "Eve", 425.00, "Processing"),
    (103, "2024-01-20", "Carol", 320.00, "Processing"),  # duplicate order_id, newer date (should keep this one)
    (106, "2024-01-20", "Frank", 280.00, "Delivered")
]

orders_columns = ["order_id", "order_date", "customer", "amount", "status"]
orders = spark.createDataFrame(orders_data, orders_columns)
orders.display()

print(f"Total orders: {orders.count()}")
print(f"Unique order_ids: {orders.select('order_id').distinct().count()}")

In [0]:

#Putting It All Together — A Realistic Example

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, count

# Step 1: know THAT duplicates exist
dup_summary = orders.groupBy("order_id").count().filter(col("count") > 1)
print("Order IDs with duplicates:", dup_summary.count())

# Step 2: know WHICH rows, choosing the most recent as the keeper
window_spec = Window.partitionBy("order_id").orderBy(col("order_date").desc())
ranked = orders.withColumn("row_num", row_number().over(window_spec))

clean_orders = ranked.filter(col("row_num") == 1).drop("row_num")
removed = ranked.filter(col("row_num") > 1)

print("Original:", orders.count(), "| Clean:", clean_orders.count(), "| Removed:", removed.count())
